# Vectorless RAG with LlamaIndex (In-Memory)

This notebook demonstrates a conversational RAG pipeline using **LlamaIndex** with an in-memory vector store (no FAISS, no disk persistence). It mirrors the LangChain + FAISS pipeline from the project's `ConversationalRAG` class.

**Key Learning Goal:** Compare LangChain vs LlamaIndex approaches for the same RAG task.

## Section 1: Package Installation

In [ ]:
%pip install llama-index-core llama-index-llms-openai llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-readers-file python-dotenv --quiet

## Section 2: Imports & Environment Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from project .env
load_dotenv("../.env")

# Verify API keys are loaded
print("Google API Key set:", bool(os.getenv("GOOGLE_API_KEY")))
print("OpenAI API Key set:", bool(os.getenv("OPENAI_API_KEY")))
print()
print("Ready to initialize LlamaIndex.")

## Section 3: Configure LlamaIndex Global Settings

LlamaIndex 0.10+ uses a global `Settings` object to configure LLM, embeddings, and chunking globally.
This is equivalent to project's `ModelLoader` + `config.yaml`.

In [ ]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.google_genai import GoogleGenerativeAIEmbedding

# === LLM Configuration ===
# Default: OpenAI gpt-3.5-turbo (matches project default via LLM_PROVIDER=openai)
Settings.llm = OpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    max_tokens=2048
)

print("LLM: OpenAI gpt-3.5-turbo")
print("  (Commented alternative: Google Gemini 2.0 Flash below)")

# === Alternative: Google Generative AI (Gemini) ===
# from llama_index.llms.google_genai import GoogleGenerativeAI
# Settings.llm = GoogleGenerativeAI(
#     model="gemini-2.0-flash",
#     temperature=0,
#     max_output_tokens=2048,
#     api_key=os.environ["GOOGLE_API_KEY"]
# )

# === Embeddings Configuration ===
# Always use Google Generative AI embeddings (matches project: models/gemini-embedding-001)
Settings.embed_model = GoogleGenerativeAIEmbedding(
    model_name="models/gemini-embedding-001",
    api_key=os.environ["GOOGLE_API_KEY"]
)

print("Embeddings: Google models/gemini-embedding-001")

# === Chunking Configuration ===
# Match project defaults: 1000 chars, 200 char overlap
Settings.chunk_size = 1000
Settings.chunk_overlap = 200

print("Chunk size: 1000 chars, overlap: 200 chars")
print("\n✓ LlamaIndex Settings configured.")

## Section 4: Load Documents

Use `SimpleDirectoryReader` to load all PDFs, DOCX, and TXT files from `data/multi_doc_chat/`.
This replaces project's `PyPDFLoader`, `Docx2txtLoader`, `TextLoader` setup.

In [ ]:
from llama_index.core import SimpleDirectoryReader

DATA_DIR = Path("../data/multi_doc_chat")

print(f"Loading documents from: {DATA_DIR.absolute()}")
print(f"Directory exists: {DATA_DIR.exists()}")

documents = SimpleDirectoryReader(
    input_dir=str(DATA_DIR),
    required_exts=[".pdf", ".docx", ".txt"],
    recursive=False
).load_data()

print(f"\n✓ Loaded {len(documents)} documents")
print("\nDocument preview (first 3):")
for i, doc in enumerate(documents[:3]):
    filename = doc.metadata.get('file_name', 'unknown')
    text_len = len(doc.text)
    print(f"  [{i+1}] {filename} | {text_len} chars")

## Section 5: Parse & Chunk Documents

Use `SentenceSplitter` (LlamaIndex's equivalent of LangChain's `RecursiveCharacterTextSplitter`) to create chunks.
Each chunk becomes a "node" in LlamaIndex.

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

parser = SentenceSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

nodes = parser.get_nodes_from_documents(documents)

print(f"✓ Total nodes (chunks) created: {len(nodes)}")
print("\nSample node (first chunk):")
print(f"  File: {nodes[0].metadata.get('file_name')}")
print(f"  Length: {len(nodes[0].text)} chars")
print(f"  Text (first 300 chars):")
print(f"  {nodes[0].text[:300]}...")

## Section 6: Build In-Memory VectorStoreIndex

Create a `VectorStoreIndex` with embeddings stored in RAM (no FAISS files, no disk persistence).
This replaces project's `FaissManager.load_or_create()` + `FAISS.from_texts()`.

In [ ]:
from llama_index.core import VectorStoreIndex

print("Building in-memory VectorStoreIndex...")
print("(This will call the embedding API for all chunks)\n")

index = VectorStoreIndex(
    nodes,
    show_progress=True
)

print("\n✓ In-memory index built successfully.")
print(f"  Vector store type: {type(index.vector_store).__name__}")
print(f"  Total indexed nodes: {len(nodes)}")

## Section 7: Create Conversational Chat Engine

Use `condense_plus_context` chat mode, which mirrors the LangChain pipeline exactly:

1. **Condense step**: Rewrites the user's query as a standalone question using conversation history
   - (Equivalent to LangChain's `contextualize_question` prompt)
2. **Context step**: Retrieves relevant documents and generates an answer
   - (Equivalent to LangChain's `context_qa` prompt)

Chat history is automatically managed by `ChatMemoryBuffer`.

In [ ]:
from llama_index.core.memory import ChatMemoryBuffer

# Create a memory buffer to track chat history
memory = ChatMemoryBuffer.from_defaults(
    token_limit=3000  # ~750 words of history
)

# Create chat engine in "condense_plus_context" mode
chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context",
    memory=memory,
    verbose=True,        # Prints condensed question + retrieved nodes
    similarity_top_k=5   # Retrieve top 5 chunks (matches project k=5)
)

print("✓ Chat engine ready.")
print(f"  Mode: condense_plus_context")
print(f"  Memory limit: 3000 tokens")
print(f"  Top-k retrieval: 5")
print("\nNext: Run a multi-turn conversation.")

## Section 8: Multi-Turn Chat Demo

Demonstrate conversational context retention across turns.
Watch how the LLM uses prior context to understand references like "it" in Turn 2.

In [ ]:
# === Turn 1: Ask about Attention mechanism ===
print("="*70)
print("TURN 1: Attention Mechanism")
print("="*70)
q1 = "What is the main idea behind the Attention mechanism?"
print(f"\nQ: {q1}")
print("-" * 70)
r1 = chat_engine.chat(q1)
print(f"\nA: {r1}")

In [ ]:
# === Turn 2: Follow-up question (tests context retention) ===
# "it" refers to Attention from Turn 1
print("\n" + "="*70)
print("TURN 2: Comparison (context-dependent)")
print("="*70)
q2 = "How does it compare to recurrent neural networks?"
print(f"\nQ: {q2}")
print("\nNote: 'it' refers to Attention from Turn 1.")
print("-" * 70)
r2 = chat_engine.chat(q2)
print(f"\nA: {r2}")

In [ ]:
# === Turn 3: Cross-document question ===
# Should retrieve from state_of_the_union.txt
print("\n" + "="*70)
print("TURN 3: Cross-Document Question")
print("="*70)
q3 = "What economic topics are mentioned in the state of the union?"
print(f"\nQ: {q3}")
print("-" * 70)
r3 = chat_engine.chat(q3)
print(f"\nA: {r3}")

## Section 9: Inspect Retrieved Source Nodes

Show which document chunks were used to answer the last query.
This helps debug retrieval quality and see relevance scores.

In [ ]:
# Create a query engine (alternative to chat engine for inspection)
query_engine = index.as_query_engine(
    similarity_top_k=5
)

print("Running a query to inspect source nodes...\n")
query = "What is multi-head attention?"
print(f"Query: {query}")
print("-" * 70)

response = query_engine.query(query)

print(f"\nAnswer: {response.response}")
print("\n" + "="*70)
print("SOURCE NODES (Retrieved Chunks)")
print("="*70)

for i, node in enumerate(response.source_nodes):
    print(f"\n[{i+1}] Relevance Score: {node.score:.4f}")
    print(f"    File: {node.metadata.get('file_name', 'unknown')}")
    print(f"    Text Preview (first 250 chars):")
    print(f"    {node.text[:250]}...")
    print("-" * 70)

## Section 10: LangChain vs LlamaIndex Comparison

Summary table showing how this notebook's approach maps to the project's LangChain pipeline.

| Concern | LangChain (Project) | LlamaIndex (This Notebook) |
|---|---|---|
| **LLM Config** | `ModelLoader().load_llm()` + config.yaml | `Settings.llm = OpenAI(...)` |
| **Embeddings** | `ModelLoader().load_embeddings()` | `Settings.embed_model = GoogleGenerativeAIEmbedding(...)` |
| **Doc Loaders** | `PyPDFLoader`, `Docx2txtLoader`, `TextLoader` | `SimpleDirectoryReader` |
| **Chunking** | `RecursiveCharacterTextSplitter(1000, 200)` | `SentenceSplitter(1000, 200)` |
| **Vector Store** | `FAISS` (disk-persisted) | `VectorStoreIndex` (in-memory) |
| **Retriever** | `vectorstore.as_retriever(k=5)` | `index.as_retriever(top_k=5)` |
| **Question Rewrite** | Custom LCEL chain → `contextualize_question` | Built into `condense_plus_context` |
| **Answer Generation** | Custom LCEL chain → `context_qa` | Built into `condense_plus_context` |
| **Chat Engine** | `ConversationalRAG.invoke(input, history)` | `chat_engine.chat(message)` |
| **Chat History** | Passed manually as `List[BaseMessage]` | Managed by `ChatMemoryBuffer` |
| **Persistence** | Indexes saved to `faiss_index/<session_id>/` | Ephemeral (lost on kernel restart) |

### Key Takeaways
- **LangChain approach**: More explicit control, manual chain assembly, disk persistence
- **LlamaIndex approach**: Higher-level abstractions, built-in modes (condense_plus_context), simpler chat management
- Both use the same Google embedding model and similar chunking strategies
- LlamaIndex's `condense_plus_context` mode combines question rewriting + retrieval + answering in one call

## Bonus Section: Persistence (Optional)

If you want to save the index for later sessions, uncomment the cells below.
(By default, the in-memory index is lost when the kernel restarts.)

In [ ]:
# # === Save index to disk ===
# PERSIST_DIR = "./vectorless_rag_index"
# os.makedirs(PERSIST_DIR, exist_ok=True)
# 
# index.storage_context.persist(persist_dir=PERSIST_DIR)
# print(f"✓ Index persisted to {PERSIST_DIR}")

In [ ]:
# # === Load index from disk (in a new notebook session) ===
# from llama_index.core import StorageContext, load_index_from_storage
# 
# PERSIST_DIR = "./vectorless_rag_index"
# storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
# index = load_index_from_storage(storage_context)
# 
# # Re-create chat engine
# memory = ChatMemoryBuffer.from_defaults(token_limit=3000)
# chat_engine = index.as_chat_engine(
#     chat_mode="condense_plus_context",
#     memory=memory,
#     verbose=True,
#     similarity_top_k=5
# )
# 
# print("✓ Index loaded from disk.")
# print("You can now chat with the documents again.")